In [1]:
"""Improved supervised PG19 six-class frequency inference.

The experiment uses a target-noun-disjoint 80/20 split. The held-out nouns are
never used for model or hyperparameter selection. Repeated executions are
combined into one feature vector per query, and candidate estimators are
compared with stratified, target-noun-grouped cross-validation on the training
partition only.

The predictors follow the grey-box configuration: instrumented RAG latencies,
execution/size observables, and query length. Frequency fields, query IDs,
target nouns, query text, semantic category, and class-encoding identifiers are
not predictors.
"""

from __future__ import annotations

import argparse
import json
import re
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "axes.titlesize": 20,
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
})

RANDOM_STATE = 42
QUERY_COLUMN = "query_id"
TARGET_COLUMN = "frequency"

FREQUENCY_ORDER = (
    "Very high",
    "High",
    "Moderate",
    "Low",
    "Very low",
    "Term not present",
)

EXPECTED_K_VALUES = (5, 20, 50, 100)
EXPECTED_ITERATIONS = (1, 2)

GREY_BOX_NUMERIC_SOURCES = (
    "rag_end_to_end_latency_ms",
    "retrieval_latency_ms",
    "llm_latency_ms",
    "prompt_formatting_latency_ms",
    "component_sum_latency_ms",
    "overhead_latency_ms",
    "latency_ms",
    "num_retrieved_docs",
    "context_num_chars",
    "prompt_num_chars",
    "answer_num_chars",
)

CATEGORICAL_FEATURES = ("length",)


def infer_target_noun_group(query_id: str) -> str:
    """Group all query templates and lengths associated with one target noun."""

    match = re.fullmatch(r"(.+)__(short|medium|long)__(.+)__(\d+)", query_id)
    if match is None:
        raise ValueError(f"Unexpected query_id format: {query_id!r}")
    frequency_key, _, category_key, suffix = match.groups()
    return f"{frequency_key}__{category_key}__target_{int(suffix) // 10}"


def validate_source(raw_df: pd.DataFrame) -> None:
    required = {
        QUERY_COLUMN,
        TARGET_COLUMN,
        "query_text",
        "length",
        "k",
        "iteration",
        *GREY_BOX_NUMERIC_SOURCES,
    }
    missing = sorted(required.difference(raw_df.columns))
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    observed_k = tuple(sorted(raw_df["k"].unique().tolist()))
    observed_iterations = tuple(sorted(raw_df["iteration"].unique().tolist()))
    if observed_k != EXPECTED_K_VALUES:
        raise ValueError(f"Expected k={EXPECTED_K_VALUES}; observed {observed_k}")
    if observed_iterations != EXPECTED_ITERATIONS:
        raise ValueError(
            f"Expected iterations={EXPECTED_ITERATIONS}; observed {observed_iterations}"
        )


def build_query_features(
    raw_df: pd.DataFrame,
) -> tuple[pd.DataFrame, list[str], list[str]]:
    """Summarize eight repeated executions into a grey-box query vector."""

    validate_source(raw_df)
    raw_df = raw_df.copy()
    raw_df["target_noun_group"] = raw_df[QUERY_COLUMN].map(infer_target_noun_group)
    raw_df["query_num_chars"] = raw_df["query_text"].astype(str).str.len()
    raw_df["query_num_words"] = (
        raw_df["query_text"].astype(str).str.split().str.len()
    )

    repetitions = raw_df.groupby(QUERY_COLUMN).size()
    expected_repetitions = len(EXPECTED_K_VALUES) * len(EXPECTED_ITERATIONS)
    if not repetitions.eq(expected_repetitions).all():
        raise ValueError(f"Each query must have {expected_repetitions} executions")

    identifiers = raw_df.groupby(QUERY_COLUMN).agg(
        frequency=(TARGET_COLUMN, "first"),
        length=("length", "first"),
        target_noun_group=("target_noun_group", "first"),
        query_num_chars=("query_num_chars", "first"),
        query_num_words=("query_num_words", "first"),
    )

    feature_blocks: list[pd.DataFrame] = [identifiers]
    generated_numeric: list[str] = ["query_num_chars", "query_num_words"]
    centered_k = np.asarray(EXPECTED_K_VALUES, dtype=float)
    centered_k -= centered_k.mean()
    slope_denominator = float(np.sum(centered_k**2))

    for source_column in GREY_BOX_NUMERIC_SOURCES:
        if raw_df[source_column].isna().all():
            raise ValueError(f"{source_column} contains only missing values")

        repeated = raw_df.pivot_table(
            index=QUERY_COLUMN,
            columns=["k", "iteration"],
            values=source_column,
            aggfunc="mean",
        )
        repeated.columns = [
            f"{source_column}__k{k_value}__iteration{iteration}"
            for k_value, iteration in repeated.columns
        ]
        generated_numeric.extend(repeated.columns.tolist())
        feature_blocks.append(repeated)

        summary = raw_df.groupby(QUERY_COLUMN)[source_column].agg(
            ["mean", "std", "min", "max", "median"]
        )
        summary.columns = [
            f"{source_column}__{statistic}" for statistic in summary.columns
        ]
        generated_numeric.extend(summary.columns.tolist())
        feature_blocks.append(summary)

        means_by_k = (
            raw_df.groupby([QUERY_COLUMN, "k"])[source_column].mean().unstack()
        ).reindex(columns=EXPECTED_K_VALUES)
        slopes = (
            (means_by_k - means_by_k.mean(axis=1).to_numpy()[:, None])
            .mul(centered_k, axis=1)
            .sum(axis=1)
            / slope_denominator
        )
        slope_name = f"{source_column}__k_slope"
        feature_blocks.append(slopes.rename(slope_name).to_frame())
        generated_numeric.append(slope_name)

    query_df = pd.concat(feature_blocks, axis=1).reset_index()

    # Retain raw values and add a signed log transform for heavy-tailed timing
    # and size variables. Both representations are derived only from permitted
    # grey-box observations.
    log_features = {
        f"signed_log1p__{column}": np.sign(query_df[column])
        * np.log1p(np.abs(query_df[column]))
        for column in generated_numeric
    }
    query_df = pd.concat(
        [query_df, pd.DataFrame(log_features, index=query_df.index)],
        axis=1,
    )
    numeric_features = generated_numeric + list(log_features)

    forbidden = {
        TARGET_COLUMN,
        "frequency_key",
        QUERY_COLUMN,
        "query_text",
        "category",
        "category_key",
        "target_noun_group",
    }
    model_features = set(CATEGORICAL_FEATURES).union(numeric_features)
    overlap = forbidden.intersection(model_features)
    if overlap:
        raise AssertionError(f"Forbidden model features detected: {sorted(overlap)}")

    noun_audit = query_df.groupby("target_noun_group").agg(
        queries=(QUERY_COLUMN, "size"),
        labels=(TARGET_COLUMN, "nunique"),
    )

    return query_df, list(CATEGORICAL_FEATURES), numeric_features


def target_noun_disjoint_split(
    query_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Hold out exactly two of ten target nouns from every frequency class."""

    rng = np.random.default_rng(RANDOM_STATE)
    test_groups: list[str] = []
    for class_name in FREQUENCY_ORDER:
        groups = np.asarray(
            sorted(
                query_df.loc[
                    query_df[TARGET_COLUMN] == class_name,
                    "target_noun_group",
                ].unique()
            )
        )
        if len(groups) != 10:
            raise ValueError(f"Expected 10 target nouns for {class_name}; found {len(groups)}")
        rng.shuffle(groups)
        test_groups.extend(groups[:2].tolist())

    is_test = query_df["target_noun_group"].isin(test_groups)
    train_df = query_df.loc[~is_test].copy().reset_index(drop=True)
    test_df = query_df.loc[is_test].copy().reset_index(drop=True)
    if not set(train_df["target_noun_group"]).isdisjoint(
        test_df["target_noun_group"]
    ):
        raise AssertionError("Target-noun leakage detected")

    audit = (
        pd.concat(
            [train_df.assign(split="train"), test_df.assign(split="test")],
            ignore_index=True,
        )
        .groupby(["split", TARGET_COLUMN], observed=False)
        .agg(
            queries=(QUERY_COLUMN, "size"),
            target_nouns=("target_noun_group", "nunique"),
        )
        .reindex(
            pd.MultiIndex.from_product(
                [["train", "test"], FREQUENCY_ORDER],
                names=["split", TARGET_COLUMN],
            )
        )
        .reset_index()
    )
    return train_df, test_df, audit


def make_preprocessor(
    categorical_features: list[str],
    numeric_features: list[str],
) -> ColumnTransformer:
    return ColumnTransformer(
        [
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                categorical_features,
            ),
            (
                "numeric",
                SimpleImputer(strategy="median"),
                numeric_features,
            ),
        ]
    )


def candidate_classifiers() -> dict[str, object]:
    """Small, predeclared candidate set evaluated without test-set access."""

    return {
        "supplied_random_forest": RandomForestClassifier(
            n_estimators=200,
            min_samples_split=15,
            min_samples_leaf=5,
            max_features="sqrt",
            max_depth=30,
            criterion="gini",
            class_weight=None,
            bootstrap=True,
            random_state=RANDOM_STATE,
            n_jobs=1,
        ),
        "balanced_entropy_random_forest": RandomForestClassifier(
            n_estimators=500,
            min_samples_split=5,
            min_samples_leaf=2,
            max_features="sqrt",
            max_depth=None,
            criterion="entropy",
            class_weight="balanced_subsample",
            bootstrap=True,
            random_state=RANDOM_STATE,
            n_jobs=1,
        ),
        "extra_trees": ExtraTreesClassifier(
            n_estimators=500,
            min_samples_split=4,
            min_samples_leaf=2,
            max_features="sqrt",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1,
        ),
    }


def grouped_model_selection(
    train_df: pd.DataFrame,
    feature_columns: list[str],
    categorical_features: list[str],
    numeric_features: list[str],
) -> tuple[str, object, pd.DataFrame]:
    """Select by grouped cross-validated macro-F1 on training data only."""

    X = train_df[feature_columns]
    y = train_df[TARGET_COLUMN].astype(str)
    groups = train_df["target_noun_group"]
    splitter = StratifiedGroupKFold(
        n_splits=4,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    preprocessor = make_preprocessor(categorical_features, numeric_features)
    candidates = candidate_classifiers()
    rows: list[dict[str, float | str]] = []

    for model_name, classifier in candidates.items():
        out_of_fold = np.empty(len(train_df), dtype=object)
        for fit_indices, validation_indices in splitter.split(X, y, groups=groups):
            pipeline = Pipeline(
                [
                    ("preprocessor", clone(preprocessor)),
                    ("classifier", clone(classifier)),
                ]
            )
            pipeline.fit(X.iloc[fit_indices], y.iloc[fit_indices])
            out_of_fold[validation_indices] = pipeline.predict(
                X.iloc[validation_indices]
            )

        rows.append(
            {
                "model": model_name,
                "cv_accuracy": accuracy_score(y, out_of_fold),
                "cv_balanced_accuracy": balanced_accuracy_score(y, out_of_fold),
                "cv_macro_f1": f1_score(
                    y,
                    out_of_fold,
                    labels=list(FREQUENCY_ORDER),
                    average="macro",
                    zero_division=0,
                ),
            }
        )

    comparison = pd.DataFrame(rows).sort_values(
        ["cv_macro_f1", "cv_accuracy"],
        ascending=False,
    )
    comparison["selected"] = False
    comparison.loc[comparison.index[0], "selected"] = True
    selected_name = str(comparison.iloc[0]["model"])
    return selected_name, candidates[selected_name], comparison.reset_index(drop=True)


def ordinal_score(true_class: str, predicted_class: str) -> float:
    positions = {label: index for index, label in enumerate(FREQUENCY_ORDER)}
    distance = abs(positions[true_class] - positions[predicted_class])
    if distance == 0:
        return 1.0
    if distance == 1:
        return 0.8
    if distance == 2:
        return 0.6
    return 0.0


def ordinal_reports(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    normal_report: pd.DataFrame,
) -> tuple[dict[str, float], pd.DataFrame]:
    ow_precision: dict[str, float] = {}
    ow_recall: dict[str, float] = {}
    ow_f1: dict[str, float] = {}

    for class_name in FREQUENCY_ORDER:
        true_predictions = y_pred[y_true == class_name]
        predicted_truths = y_true[y_pred == class_name]
        ow_recall[class_name] = float(
            np.mean([ordinal_score(class_name, pred) for pred in true_predictions])
        )
        ow_precision[class_name] = (
            float(
                np.mean(
                    [ordinal_score(true, class_name) for true in predicted_truths]
                )
            )
            if len(predicted_truths)
            else np.nan
        )
        precision = ow_precision[class_name]
        recall = ow_recall[class_name]
        if np.isnan(precision) or np.isnan(recall):
            ow_f1[class_name] = np.nan
        elif precision + recall == 0:
            ow_f1[class_name] = 0.0
        else:
            ow_f1[class_name] = float(
                2 * precision * recall / (precision + recall)
            )

    report = pd.DataFrame(
        {
            "OW Precision": [ow_precision[name] for name in FREQUENCY_ORDER],
            "OW Recall": [ow_recall[name] for name in FREQUENCY_ORDER],
            "OW F1": [ow_f1[name] for name in FREQUENCY_ORDER],
            "support": [normal_report.loc[name, "support"] for name in FREQUENCY_ORDER],
        },
        index=FREQUENCY_ORDER,
    )
    metrics = {
        "ow_accuracy": float(
            np.mean(
                [
                    ordinal_score(true, predicted)
                    for true, predicted in zip(y_true, y_pred)
                ]
            )
        ),
        "macro_ow_precision": float(np.nanmean(list(ow_precision.values()))),
        "macro_ow_recall": float(np.nanmean(list(ow_recall.values()))),
        "macro_ow_f1": float(np.nanmean(list(ow_f1.values()))),
    }
    return metrics, report


def normal_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=list(FREQUENCY_ORDER),
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=list(FREQUENCY_ORDER),
                average="weighted",
                zero_division=0,
            )
        ),
    }


def run_experiment(csv_path: Path, output_directory: Path) -> dict:
    raw_df = pd.read_csv(csv_path)
    query_df, categorical_features, numeric_features = build_query_features(raw_df)
    feature_columns = categorical_features + numeric_features
    train_df, test_df, split_audit = target_noun_disjoint_split(query_df)

    selected_name, selected_classifier, selection = grouped_model_selection(
        train_df,
        feature_columns,
        categorical_features,
        numeric_features,
    )
    preprocessor = make_preprocessor(categorical_features, numeric_features)
    model = Pipeline(
        [
            ("preprocessor", preprocessor),
            ("classifier", clone(selected_classifier)),
        ]
    )
    model.fit(train_df[feature_columns], train_df[TARGET_COLUMN])
    y_true = test_df[TARGET_COLUMN].to_numpy()
    y_pred = model.predict(test_df[feature_columns])

    baseline = Pipeline(
        [
            ("preprocessor", make_preprocessor(categorical_features, numeric_features)),
            ("classifier", candidate_classifiers()["supplied_random_forest"]),
        ]
    )
    baseline.fit(train_df[feature_columns], train_df[TARGET_COLUMN])
    baseline_pred = baseline.predict(test_df[feature_columns])

    normal = normal_metrics(y_true, y_pred)
    baseline_normal = normal_metrics(y_true, baseline_pred)
    normal_report = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=list(FREQUENCY_ORDER),
            output_dict=True,
            zero_division=0,
        )
    ).T.loc[
        list(FREQUENCY_ORDER),
        ["precision", "recall", "f1-score", "support"],
    ]
    ordinal, ordinal_report = ordinal_reports(y_true, y_pred, normal_report)

    matrix = confusion_matrix(y_true, y_pred, labels=list(FREQUENCY_ORDER))
    matrix_df = pd.DataFrame(
        matrix,
        index=pd.Index(FREQUENCY_ORDER, name="True frequency"),
        columns=pd.Index(FREQUENCY_ORDER, name="Predicted frequency"),
    )

    output_directory.mkdir(parents=True, exist_ok=True)
    selection.to_csv(output_directory / "model_selection.csv", index=False)
    split_audit.to_csv(output_directory / "split_audit.csv", index=False)
    normal_report.to_csv(output_directory / "normal_classification_report.csv")
    ordinal_report.to_csv(
        output_directory / "ordinal_weighted_classification_report.csv"
    )
    matrix_df.to_csv(output_directory / "confusion_matrix.csv")
    test_df[[QUERY_COLUMN, "target_noun_group", TARGET_COLUMN]].assign(
        predicted_frequency=y_pred
    ).to_csv(output_directory / "held_out_predictions.csv", index=False)

    transformed_names = model.named_steps["preprocessor"].get_feature_names_out()
    classifier = model.named_steps["classifier"]
    if hasattr(classifier, "feature_importances_"):
        feature_importance = pd.DataFrame(
            {
                "feature": transformed_names,
                "importance": classifier.feature_importances_,
            }
        ).sort_values("importance", ascending=False)
        feature_importance.to_csv(
            output_directory / "feature_importance.csv", index=False
        )

    fig, ax = plt.subplots(figsize=(10.5, 9.0))
    ConfusionMatrixDisplay(
        confusion_matrix=matrix,
        display_labels=list(FREQUENCY_ORDER),
    ).plot(
        ax=ax,
        cmap="Blues",
        xticks_rotation=35,
        colorbar=False,
        values_format="d",
    )
    ax.set_title("Pubmed qwen supervised white-box frequency inference", pad=12)
    ax.set_xlabel("Predicted frequency")
    ax.set_ylabel("True frequency")
    for text in ax.texts:
        text.set_fontsize(24)
    for label in ax.get_xticklabels():
        label.set_ha("right")
        label.set_rotation_mode("anchor")
    fig.tight_layout()
    fig.savefig(output_directory / "confusion_matrix.png", dpi=300)
    fig.savefig(output_directory / "confusion_matrix.pdf", bbox_inches="tight")
    plt.close(fig)

    result = {
        "selected_model": selected_name,
        "selection_rule": "highest grouped-CV macro F1 on training nouns",
        "train_queries": len(train_df),
        "test_queries": len(test_df),
        "train_target_nouns": int(train_df["target_noun_group"].nunique()),
        "test_target_nouns": int(test_df["target_noun_group"].nunique()),
        "categorical_features": categorical_features,
        "numeric_feature_count": len(numeric_features),
        "normal_metrics": normal,
        "ordinal_weighted_metrics": ordinal,
        "matched_supplied_baseline_metrics": baseline_normal,
    }
    with (output_directory / "metrics.json").open("w", encoding="utf-8") as handle:
        json.dump(result, handle, indent=2)

    print("\nMODEL SELECTION (TRAINING NOUNS ONLY)")
    print(selection.round(6).to_string(index=False))
    print("\nSelected model:", selected_name)
    print("Train/test queries:", len(train_df), len(test_df))
    print("Train/test target nouns:", train_df.target_noun_group.nunique(), test_df.target_noun_group.nunique())
    print("\nNORMAL METRICS")
    for name, value in normal.items():
        print(f"{name}: {value:.6f}")
    print("\nORDINAL WEIGHTED METRICS")
    for name, value in ordinal.items():
        print(f"{name}: {value:.6f}")
    print("\nMATCHED SUPPLIED-RF BASELINE")
    for name, value in baseline_normal.items():
        print(f"{name}: {value:.6f}")
    print("\nNORMAL CLASSIFICATION REPORT")
    print(normal_report.round(4).to_string())
    print("\nORDINAL WEIGHTED CLASSIFICATION REPORT")
    print(ordinal_report.round(4).to_string())
    print(f"\nResults written to: {output_directory.resolve()}")
    return result


def parse_arguments() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--csv",
        type=Path,
        default=Path("per_query_rag_llm_latencies_ordered_qwen_pubmed.csv"),
    )
    parser.add_argument(
        "--output",
        type=Path,
        default=Path("pubmed_qwen_white_box_freq_results"),
    )
    # parse_known_args (instead of parse_args) discards arguments it does not
    # recognize, such as the "--f=/path/to/kernel-....json" flag that Jupyter
    # injects into sys.argv when this script is run inside a notebook cell.
    # Without this, argparse raises SystemExit on that flag.
    arguments, _unrecognized = parser.parse_known_args()
    return arguments


if __name__ == "__main__":
    arguments = parse_arguments()
    run_experiment(arguments.csv, arguments.output)


MODEL SELECTION (TRAINING NOUNS ONLY)
                         model  cv_accuracy  cv_balanced_accuracy  cv_macro_f1  selected
balanced_entropy_random_forest     0.642657              0.642344     0.633729      True
                   extra_trees     0.631469              0.631104     0.619221     False
        supplied_random_forest     0.627273              0.626961     0.615986     False

Selected model: balanced_entropy_random_forest
Train/test queries: 1430 360
Train/test target nouns: 48 12

NORMAL METRICS
accuracy: 0.672222
balanced_accuracy: 0.672222
macro_f1: 0.671164
weighted_f1: 0.671164

ORDINAL WEIGHTED METRICS
ow_accuracy: 0.873333
macro_ow_precision: 0.874941
macro_ow_recall: 0.873333
macro_ow_f1: 0.873646

MATCHED SUPPLIED-RF BASELINE
accuracy: 0.655556
balanced_accuracy: 0.655556
macro_f1: 0.653009
weighted_f1: 0.653009

NORMAL CLASSIFICATION REPORT
                  precision  recall  f1-score  support
Very high            0.8000  0.8667    0.8320     60.0
High      